# 10. 라오스 EV·모빌리티 뉴스 점유율(Share of Voice) 수집·분석

**목적**: 라오스 현지 정량 근거 보강 — 임원진 벤치마크(LOCA EV)를 데이터로 독립 검증 + 실제 경쟁사 발굴 + KOKKOK Move 현지 인지도 파악.
**한계**: 라오어(lo)는 Google News 색인 부재로 0건(영/한 중심). Facebook은 Meta API 제약으로 미수집(P2). 브랜드 언급 절대량 작음 → 점유율·인지도 프록시로 해석.
**결과**: [`outputs/laos_market_share_of_voice.md`](../outputs/laos_market_share_of_voice.md)

In [ ]:
import sys, os, requests, feedparser, time, re; sys.path.append('..'); sys.path.insert(0,'../src'); sys.path.insert(0,'src')
import pandas as pd
from dotenv import load_dotenv; load_dotenv('../.env')
from db import get_connection, insert_df
def clean(t): return re.sub('<[^>]+>','',t or '').strip()

## 1. Google News RSS (en) + Naver (ko) — 라오스 모빌리티/EV 초점 쿼리

In [ ]:
G_QUERIES = [('KOKKOK Move Laos','en'),('KOKKOK Laos','en'),('LOCA Laos','en'),
 ('LOCA EV charging Laos','en'),('LOCA taxi Laos','en'),('Vientiane EV charging','en'),
 ('Laos EV charging station','en'),('Laos electric vehicle','en'),('Laos EV adoption','en'),
 ('electric motorcycle Laos','en'),('Xanh SM Laos','en'),('VinFast Laos','en'),
 ('Grab Laos ride','en'),('Laos ride hailing app','en'),('Vientiane taxi app','en'),
 ('ລົດໄຟຟ້າ ລາວ','lo'),('LOCA ລາວ','lo')]   # lo는 색인 거의 없음(한계)
gl={'en':'US','lo':'LA'}; rows=[]
for q,lang in G_QUERIES:
    url=f"https://news.google.com/rss/search?q={requests.utils.quote(q)}&hl={lang}&gl={gl[lang]}&ceid={gl[lang]}:{lang}"
    feed=feedparser.parse(url)
    for e in feed.entries:
        pdd=time.strftime('%Y-%m-%d',e.published_parsed) if e.get('published_parsed') else None
        src=e.get('source',{})
        rows.append({'source':'google_news','publisher':src.get('title','') if isinstance(src,dict) else '',
          'title':clean(e.get('title','')),'content':clean(e.get('summary','')),'url':e.get('link',''),
          'published_date':pdd,'country':'LAO','category':'market','lang_detected':lang})
    time.sleep(0.4)
print('Google:',len(rows))
# --- 정책 기조 전용 쿼리 (category='policy'로 적재) ---
POLICY_QUERIES = [('Laos ban internal combustion vehicles','en'),('Laos EV tax exemption','en'),
 ('Laos EV policy 2030','en'),('Laos electric vehicle incentive subsidy','en'),
 ('Laos government EV target','en'),('Laos green mobility policy','en'),
 ('Laos EV price control','en'),('Laos electric vehicle workforce training','en'),
 ('라오스 내연기관 수입 중단',''),('라오스 전기차 정책',''),('라오스 전기차 세금 면제','')]
# 위 쿼리는 G_QUERIES와 동일 방식으로 수집하되 category='policy'로 저장.
# 주의: 글로벌·ASEAN 정책 뉴스 노이즈가 섞이므로 라오스 미언급분은 country='OTH'로 재태깅(데이터 위생).


In [ ]:
CID,CS=os.getenv('NAVER_CLIENT_ID'),os.getenv('NAVER_CLIENT_SECRET')
from email.utils import parsedate_to_datetime
def naver(q,mx=60):
    res,start=[],1
    while len(res)<mx:
        d=requests.get('https://openapi.naver.com/v1/search/news.json',
          headers={'X-Naver-Client-Id':CID,'X-Naver-Client-Secret':CS},
          params={'query':q,'display':min(100,mx-len(res)),'start':start,'sort':'date'},timeout=10).json()
        it=d.get('items',[])
        if not it: break
        res+=it; start+=len(it)
        if len(it)<100: break
        time.sleep(0.3)
    return res
for q in ['라오스 전기차','라오스 EV 충전','비엔티안 전기차','KOKKOK 라오스','라오스 LOCA 전기차']:
    for x in naver(q):
        try: pdd=parsedate_to_datetime(x.get('pubDate','')).strftime('%Y-%m-%d')
        except: pdd=None
        rows.append({'source':'naver_news','publisher':'','title':clean(x.get('title','')),
          'content':clean(x.get('description','')),'url':x.get('originallink') or x.get('link',''),
          'published_date':pdd,'country':'LAO','category':'market','lang_detected':'ko'})
print('합계:',len(rows))

## 2. 중복 제거 후 적재

In [ ]:
df=pd.DataFrame(rows).drop_duplicates('url')
conn=get_connection(); ex=set(pd.read_sql("SELECT url FROM news_articles WHERE url IS NOT NULL",conn).url.dropna())
df=df[~df.url.isin(ex)]
print('신규',len(df),'건'); insert_df(df,'news_articles')

## 3. 점유율 분석 — EV/모빌리티 관련만, 브랜드 분류

In [ ]:
d=pd.read_sql("SELECT title,content,published_date FROM news_articles WHERE country='LAO'",conn); conn.close()
d['t']=(d.title.fillna('')+' '+d.content.fillna('')).str.lower()
d=d[d.t.str.contains(r'laos|lao\b|vientiane|라오스|비엔티안',regex=True)]
d=d[d.t.str.contains(r'\bev\b|electric|전기차|xe điện|충전|charg|mobility|모빌리티|taxi|ride|택시|kokkok|loca|xanh|vinfast|grab',regex=True)]
def player(t):
    if re.search(r'kokkok',t): return 'KOKKOK Move'
    if re.search(r'\bloca\b',t): return 'LOCA'
    if re.search(r'xanh sm|vinfast|green and smart|gsm ',t): return 'VinFast·Xanh SM'
    if re.search(r'\bgrab\b',t): return 'Grab'
    return None
d['player']=d.t.apply(player)
print('EV/모빌리티 뉴스',len(d),'| 브랜드 언급')
print(d.dropna(subset=['player']).player.value_counts().to_string())
d['yr']=pd.to_datetime(d.published_date,errors='coerce').dt.year
print('\n연도별:'); print(d[d.yr>=2022].groupby('yr').size().to_string())

## 결론
- **점유율(브랜드 41건)**: VinFast·Xanh SM 21 > LOCA 17 > KOKKOK Move 3 > Grab 0
- **뉴스량**: 2022(10) → 2026(132), 13배 — EV 모멘텀 확인
- **벤치마크**: LOCA(현지 1위, 데이터로 검증) + **VinFast·Xanh SM(놓쳤던 EV 라이드헤일링 신규 강자)** 2축
- **Grab 미진출**(0) → 경쟁군 축소. **KOKKOK Move 인지도 약세**(3) → 차별화·마케팅 필요
- 차트: `outputs/19_laos_share_of_voice.png`, `outputs/20_laos_news_trend.png`